# iFood Customer Segmentation Analysis

본 노트북은 iFood 고객 데이터를 기반으로  
연령별 구매 채널 선호, 할인 민감도, 캠페인 응답 여부에 따른 고객 특성 차이를 분석한다.

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path("..") if (Path("..") / "data").exists() else Path(".")
DATA_PATH = BASE_DIR / "data" / "marketing_campaign.csv"
FIG_DIR = BASE_DIR / "reports" / "figures"
TABLE_DIR = BASE_DIR / "reports" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")

# (Windows) 한글 폰트가 있으면 적용
try:
    plt.rcParams["font.family"] = "Malgun Gothic"
except Exception:
    pass

plt.rcParams["axes.unicode_minus"] = False

PALETTE = {
    "NumWebPurchases": "#4C72B0",
    "NumCatalogPurchases": "#DD8452",
    "NumStorePurchases": "#55A868",
    "NumDealsPurchases": "#C44E52",
}

def savefig(filename: str, dpi: int = 160):
    """현재 figure를 reports/figures/filename 으로 저장"""
    out = FIG_DIR / filename
    plt.tight_layout()
    plt.savefig(out, dpi=dpi, bbox_inches="tight")
    plt.close()
    return out

def annotate_bars(ax, fmt="{:.2f}", fontsize=9, rotation=0):
    """Seaborn/matplotlib barplot 위에 값을 표시"""
    for p in ax.patches:
        h = p.get_height()
        if pd.isna(h):
            continue
        ax.annotate(
            fmt.format(h),
            (p.get_x() + p.get_width() / 2, h),
            ha="center", va="bottom",
            fontsize=fontsize,
            rotation=rotation,
            xytext=(0, 4),
            textcoords="offset points"
        )

In [26]:
ifood = pd.read_csv(DATA_PATH, sep="\t")

print("Shape:", ifood.shape)
display(ifood.head())

Shape: (2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [27]:
ifood.iloc[:,:10].describe()

,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines
count,2240.000000,2240.000000,2216.000000,2240.000000,2240.000000,2240.000000,2240.000000
mean,5592.159821,1968.805804,52247.251354,0.444196,0.506250,49.109375,303.935714
std,3246.662198,11.984069,25173.076661,0.538398,0.544538,28.962453,336.597393
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,0.000000,0.000000
25%,2828.250000,1959.000000,35303.000000,0.000000,0.000000,24.000000,23.750000
50%,5458.500000,1970.000000,51381.500000,0.000000,0.000000,49.000000,173.500000
75%,8427.750000,1977.000000,68522.000000,1.000000,1.000000,74.000000,504.250000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,99.000000,1493.000000


## Feature Engineering (파생 변수 생성)

아래 파생 변수를 생성합니다.

- `Age` = 2014 - Year_Birth  
- `AgeGroup` : 20s(최소나이가 18세이므로 18 이상)/30s/40s/50s/60s+  
- `TotalChildren` = Kidhome + Teenhome  
- `IncomeGroup` : Income 4분위(low ~ high)  
- `CampaignResponse` : (AcceptedCmp1~5 + Response) 합이 1 이상이면 `responded`, 아니면 `not_responded`  
- `TotalPurchases` : Web+Catalog+Store+Deals 구매 횟수 합  
- `DiscountRatio` : Deals 구매 비중  
- `TotalSpend` : Mnt* 구매금액 합


In [28]:
# -----------------------------
# Feature engineering
# -----------------------------
ifood["Age"] = 2014 - ifood["Year_Birth"].astype(int)

bins = [18, 30, 40, 50, 60, 100]
labels = ["20s", "30s", "40s", "50s", "60s+"]
ifood["AgeGroup"] = pd.cut(ifood["Age"], bins=bins, labels=labels, right=False)
ifood["TotalChildren"] = ifood["Kidhome"] + ifood["Teenhome"]
ifood["IncomeGroup"] = pd.qcut(ifood["Income"], 4, labels=["low", "low_middle", "middle_high", "high"])

campaign_cols = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5", "Response"]
ifood["CampaignResponse"] = (
    ifood[campaign_cols]
    .sum(axis=1)
    .apply(lambda x: "responded" if x > 0 else "not_responded")
)

ifood["TotalPurchases"] = ifood[
    ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases", "NumDealsPurchases"]
].sum(axis=1)

ifood["DiscountRatio"] = ifood["NumDealsPurchases"] / ifood["TotalPurchases"].replace(0, np.nan)

ifood["TotalSpend"] = ifood[
    ["MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds"]
].sum(axis=1)

display(ifood[[
    "Year_Birth","Age","AgeGroup",
    "Kidhome","Teenhome","TotalChildren",
    "Income","IncomeGroup",
    "CampaignResponse",
    "TotalPurchases","DiscountRatio","TotalSpend"
]].head())

,Year_Birth,Age,AgeGroup,Kidhome,Teenhome,TotalChildren,Income,IncomeGroup,CampaignResponse,TotalPurchases,DiscountRatio,TotalSpend
0,1957,57,50s,0,0,0,58138.0,middle_high,responded,25,0.120000,1617
1,1954,60,60s+,1,1,2,46344.0,low_middle,not_responded,6,0.333333,27
2,1965,49,40s,0,0,0,71613.0,high,not_responded,21,0.047619,776
3,1984,30,30s,1,0,1,26646.0,low,not_responded,8,0.250000,53
4,1981,33,30s,1,0,1,58293.0,middle_high,not_responded,19,0.263158,422


## 요약 테이블 생성 & CSV Export

생성되는 테이블:

- `age_channel_mean.csv` : 연령대별 평균 채널 이용 횟수  
- `age_channel_share.csv` : 연령대별 평균 채널 이용 **비중(share)**  
- `children_discount_ratio.csv` : 자녀 수별 할인 구매 비중 평균  
- `income_discount_ratio.csv` : 소득 그룹별 할인 구매 비중 평균  
- `response_summary.csv` : 캠페인 응답 여부별 요약(평균/표본수)


In [29]:
# -----------------------------
# Table exports
# -----------------------------
age_channel_mean = (
    ifood
    .groupby("AgeGroup")[["NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumDealsPurchases"]]
    .mean()
    .reset_index()
)

age_channel_share = (
    ifood
    .groupby("AgeGroup")[["NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumDealsPurchases"]]
    .mean()
)
age_channel_share = age_channel_share.div(age_channel_share.sum(axis=1), axis=0).reset_index()

children_discount = (
    ifood.groupby("TotalChildren", dropna=False)["DiscountRatio"]
    .mean()
    .reset_index()
)

income_discount = (
    ifood.groupby("IncomeGroup", dropna=False)["DiscountRatio"]
    .mean()
    .reset_index()
)

response_summary = (
    ifood.groupby("CampaignResponse")
    .agg(
        total_spend_mean=("TotalSpend","mean"),
        web_visits_mean=("NumWebVisitsMonth","mean"),
        discount_ratio_mean=("DiscountRatio","mean"),
        n=("CampaignResponse","size")
    )
    .reset_index()
)

age_channel_mean.to_csv(TABLE_DIR / "age_channel_mean.csv", index=False)
age_channel_share.to_csv(TABLE_DIR / "age_channel_share.csv", index=False)
children_discount.to_csv(TABLE_DIR / "children_discount_ratio.csv", index=False)
income_discount.to_csv(TABLE_DIR / "income_discount_ratio.csv", index=False)
response_summary.to_csv(TABLE_DIR / "response_summary.csv", index=False)

display(age_channel_mean)
display(age_channel_share)
display(children_discount)
display(income_discount)
display(response_summary)

print("Saved tables to:", TABLE_DIR.resolve())


C:\Users\poulj\AppData\Local\Temp\ipykernel_32640\1406082398.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("AgeGroup")[["NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumDealsPurchases"]]
C:\Users\poulj\AppData\Local\Temp\ipykernel_32640\1406082398.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("AgeGroup")[["NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumDealsPurchases"]]
C:\Users\poulj\AppData\Local\Temp\ipykernel_32640\1406082398.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pan

,AgeGroup,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumDealsPurchases
0,20s,3.352941,2.520362,5.515837,1.651584
1,30s,3.739677,2.292639,5.310592,2.226212
2,40s,4.121302,2.329882,5.579882,2.554734
3,50s,4.411135,3.154176,6.327623,2.511777
4,60s+,4.661392,3.398734,6.512658,2.215190


,AgeGroup,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumDealsPurchases
0,20s,0.257113,0.193269,0.422970,0.126648
1,30s,0.275602,0.168960,0.391373,0.164065
2,40s,0.282556,0.159736,0.382556,0.175152
3,50s,0.268894,0.192273,0.385720,0.153113
4,60s+,0.277663,0.202451,0.387936,0.131951


,TotalChildren,DiscountRatio
0,0,0.080264
1,1,0.190115
2,2,0.291905
3,3,0.302541


,IncomeGroup,DiscountRatio
0,low,0.261650
1,low_middle,0.237704
2,middle_high,0.155257
3,high,0.067584
4,NaN,0.196551


,CampaignResponse,total_spend_mean,web_visits_mean,discount_ratio_mean,n
0,not_responded,458.109135,5.421827,0.195079,1631
1,responded,1001.333333,5.034483,0.142465,609


Saved tables to: C:\Users\poulj\Desktop\현생살자\ifood-customer-sepmentation-analysis\reports\tables


## 시각화 생성 (슬라이드 스타일 유지)

### Figure 01 — 연령대별 평균 채널 이용 비중 (파이 차트)

- 각 연령대(20s~60s+)별로 1개의 파이 차트를 그리고,  
- 채널(Web/Catalog/Store/Deals)의 비중을 `autopct`로 표시합니다.

✅ 저장 파일: `reports/figures/01_age_channel_share_pies.png`


In [30]:
# -----------------------------
# Figure 01: AgeGroup share pies (same logic as slide)
# -----------------------------
fig, axes = plt.subplots(1, len(labels), figsize=(18, 4))

for ax, age in zip(axes, labels):
    row = age_channel_share[age_channel_share["AgeGroup"] == age]
    if row.empty:
        ax.axis("off")
        continue

    vals = row[["NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumDealsPurchases"]].iloc[0].values

    ax.pie(
        vals,
        autopct="%1.1f%%",
        startangle=90,
        colors=[
            PALETTE["NumWebPurchases"],
            PALETTE["NumCatalogPurchases"],
            PALETTE["NumStorePurchases"],
            PALETTE["NumDealsPurchases"],
        ],
        textprops={"fontsize": 9}
    )
    ax.set_title(str(age), fontsize=10)

fig.suptitle("연령대별 평균 채널 이용 비중", fontsize=14)
fig.legend(
    ["NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumDealsPurchases"],
    loc="lower center",
    ncol=4,
    frameon=False
)

out = savefig("01_age_channel_share_pies.png")
print("Saved:", out.resolve())


Saved: C:\Users\poulj\Desktop\현생살자\ifood-customer-sepmentation-analysis\reports\figures\01_age_channel_share_pies.png


### Figure 02 — 연령대별 평균 채널 이용 횟수 (묶은 막대)

- 연령대별 평균 구매 횟수를 채널별로 묶어서(bar + hue) 표현합니다.
- 막대 위에 값(레이블)을 표시합니다.

✅ 저장 파일: `reports/figures/02_age_channel_avg_counts.png`


In [31]:
# -----------------------------
# Figure 02: AgeGroup average channel counts (grouped bar)
# -----------------------------
age_channel_long = pd.melt(
    age_channel_mean,
    id_vars="AgeGroup",
    var_name="Channel",
    value_name="AvgPurchases"
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x="AgeGroup",
    y="AvgPurchases",
    hue="Channel",
    data=age_channel_long,
    palette=[
        PALETTE["NumWebPurchases"],
        PALETTE["NumCatalogPurchases"],
        PALETTE["NumStorePurchases"],
        PALETTE["NumDealsPurchases"],
    ]
)

plt.title("연령대별 평균 채널 이용 횟수")
plt.xlabel("연령대")
plt.ylabel("평균 구매 횟수")
plt.legend(title="구매 채널")

annotate_bars(ax, fmt="{:.2f}", fontsize=8)

out = savefig("02_age_channel_avg_counts.png")
print("Saved:", out.resolve())


c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:1273: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregator, agg_var)
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to ge

Saved: C:\Users\poulj\Desktop\현생살자\ifood-customer-sepmentation-analysis\reports\figures\02_age_channel_avg_counts.png


### Figure 03 — 자녀 수별 할인 구매 비중 (막대)

✅ 저장 파일: `reports/figures/03_children_discount_ratio_bar.png`

In [32]:
# -----------------------------
# Figure 03: Children discount ratio (bar)
# -----------------------------
plt.figure(figsize=(8, 5))
ax = sns.barplot(x="TotalChildren", y="DiscountRatio", data=children_discount, color="#4C72B0")

plt.title("자녀 수별 할인 구매 비중")
plt.xlabel("자녀 수")
plt.ylabel("할인 구매 비중")

annotate_bars(ax, fmt="{:.3f}")

out = savefig("03_children_discount_ratio_bar.png")
print("Saved:", out.resolve())


c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:1273: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregator, agg_var)


Saved: C:\Users\poulj\Desktop\현생살자\ifood-customer-sepmentation-analysis\reports\figures\03_children_discount_ratio_bar.png


### Figure 04 — 소득 수준별 할인 구매 비중 (막대)

- 소득 그룹은 `low → low_middle → middle_high → high` 순서로 표시합니다.

✅ 저장 파일: `reports/figures/04_income_discount_ratio_bar.png`

In [33]:
# -----------------------------
# Figure 04: Income discount ratio (bar)
# -----------------------------
income_order = ["low", "low_middle", "middle_high", "high"]

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x="IncomeGroup",
    y="DiscountRatio",
    data=income_discount,
    order=income_order,
    color="#55A868"
)

plt.title("소득 수준별 할인 구매 비중")
plt.xlabel("소득 수준")
plt.ylabel("할인 구매 비중")

annotate_bars(ax, fmt="{:.3f}")

out = savefig("04_income_discount_ratio_bar.png")
print("Saved:", out.resolve())


c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:1273: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregator, agg_var)


Saved: C:\Users\poulj\Desktop\현생살자\ifood-customer-sepmentation-analysis\reports\figures\04_income_discount_ratio_bar.png


### Figure 05~07 — 캠페인 응답 여부별 분포 (박스플롯)

- 05: 총 구매금액(`TotalSpend`)
- 06: 할인 구매 비중(`DiscountRatio`)
- 07: 월간 웹사이트 방문 수(`NumWebVisitsMonth`)

✅ 저장 파일:
- `05_response_total_spend_box.png`
- `06_response_discount_ratio_box.png`
- `07_response_web_visits_box.png`


In [34]:
# -----------------------------
# Figure 05: Response total spend (boxplot)
# -----------------------------
resp_order = ["responded", "not_responded"]

plt.figure(figsize=(6, 5))
sns.boxplot(
    x="CampaignResponse",
    y="TotalSpend",
    data=ifood,
    order=resp_order,
    palette="muted"
)
plt.title("캠페인 응답 여부별 총 구매 금액 분포")
plt.xlabel("캠페인 응답 여부")
plt.ylabel("총 구매 금액")

out = savefig("05_response_total_spend_box.png")
print("Saved:", out.resolve())


Saved: C:\Users\poulj\Desktop\현생살자\ifood-customer-sepmentation-analysis\reports\figures\05_response_total_spend_box.png


C:\Users\poulj\AppData\Local\Temp\ipykernel_32640\2777774698.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:632: FutureWarning: SeriesGroupBy.grouper is deprecated and will be removed in a future version of pandas.
  positions = grouped.grouper.result_index.to_numpy(dtype=float)
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with

In [35]:
# -----------------------------
# Figure 06: Response discount ratio (boxplot)
# -----------------------------
plt.figure(figsize=(6, 5))
sns.boxplot(
    x="CampaignResponse",
    y="DiscountRatio",
    data=ifood,
    order=resp_order,
    palette="muted"
)
plt.title("캠페인 응답 여부별 할인 구매 비중 분포")
plt.xlabel("캠페인 응답 여부")
plt.ylabel("할인 구매 비중")

out = savefig("06_response_discount_ratio_box.png")
print("Saved:", out.resolve())


C:\Users\poulj\AppData\Local\Temp\ipykernel_32640\1822178046.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:632: FutureWarning: SeriesGroupBy.grouper is deprecated and will be removed in a future version of pandas.
  positions = grouped.grouper.result_index.to_numpy(dtype=float)
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with

Saved: C:\Users\poulj\Desktop\현생살자\ifood-customer-sepmentation-analysis\reports\figures\06_response_discount_ratio_box.png


In [36]:
# -----------------------------
# Figure 07: Response web visits (boxplot)
# -----------------------------
plt.figure(figsize=(6, 5))
sns.boxplot(
    x="CampaignResponse",
    y="NumWebVisitsMonth",
    data=ifood,
    order=resp_order,
    palette="pastel"
)
plt.title("캠페인 응답 여부별 월간 웹사이트 방문 수 분포")
plt.xlabel("캠페인 응답 여부")
plt.ylabel("월간 웹사이트 방문 수")

out = savefig("07_response_web_visits_box.png")
print("Saved:", out.resolve())

C:\Users\poulj\AppData\Local\Temp\ipykernel_32640\2536925020.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:632: FutureWarning: SeriesGroupBy.grouper is deprecated and will be removed in a future version of pandas.
  positions = grouped.grouper.result_index.to_numpy(dtype=float)
c:\Users\poulj\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with

Saved: C:\Users\poulj\Desktop\현생살자\ifood-customer-sepmentation-analysis\reports\figures\07_response_web_visits_box.png
